# PK Figure to CSV

Recover the underlying numbers from a published pharmacokinetic **concentration-time plot** by
clicking the observed data points, and save them as a tidy CSV that can be re-plotted or fed
into a PK model.

This is a *manual* extractor: you tell it where the axis ticks are and where the markers are.
Nothing is guessed, so the result is only as good as your clicks — but it works on any figure,
including noisy scans, overlapping curves and error bars.

## Workflow

| Step | What you do |
|---|---|
| 1. Configure | Point at an image and declare whether each axis is linear or log |
| 2. Calibrate | Click two known ticks on the x-axis, then two on the y-axis |
| 3. Extract | For each curve, click the centre of every marker (right-click undoes) |
| 4. Save | Write `trajectory, x, y` to `extracted_data/<image name>.csv` |
| 5. Verify | Re-plot from the CSV and compare against the original figure |

## Requirements

Interactive clicking needs a live matplotlib backend. With Jupyter Lab / Notebook, install
`ipympl` and keep the `%matplotlib widget` magic below:

```
pip install numpy pandas matplotlib pillow ipympl
```

If clicks do not register, swap the magic for `%matplotlib qt` or `%matplotlib tk` to get a
separate plot window, and restart the kernel.

In [ ]:
%matplotlib widget
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

## The extractor

A small state machine around a matplotlib figure: `calibrate` records two reference clicks per
axis, `start_trajectory` / `finish_trajectory` collect the observed points for one curve, and
`save_csv` / `replot` produce the final artefacts.

Pixel-to-data conversion handles linear and log axes. For log axes, both reference values must
be strictly positive.

In [ ]:
class PKFigureExtractor:
    """Click-based extractor for time-concentration plots."""

    def __init__(self, image_path, x_log=False, y_log=False):
        self.image_path = Path(image_path)
        self.img = np.array(Image.open(self.image_path).convert("RGB"))
        self.x_log = x_log
        self.y_log = y_log
        self.x_refs = None  # [(pixel_x, data_x), (pixel_x, data_x)]
        self.y_refs = None  # [(pixel_y, data_y), (pixel_y, data_y)]
        self.trajectories = {}  # name -> list of (x_data, y_data)
        self._fig = None
        self._ax = None
        self._cid = None
        self._mode = None
        self._pending = []
        self._pending_meta = None

    # ---------- figure helpers ----------
    def _open(self, title):
        if self._fig is not None:
            plt.close(self._fig)
        self._fig, self._ax = plt.subplots(figsize=(11, 8))
        self._ax.imshow(self.img)
        self._ax.set_xticks([]); self._ax.set_yticks([])
        self._ax.set_title(title, fontsize=11)
        return self._fig, self._ax

    def _connect(self, handler):
        if self._cid is not None and self._fig is not None:
            self._fig.canvas.mpl_disconnect(self._cid)
        self._cid = self._fig.canvas.mpl_connect("button_press_event", handler)

    # ---------- coordinate conversion ----------
    def pixel_to_data(self, px, py):
        if self.x_refs is None or self.y_refs is None:
            raise RuntimeError("Axes not calibrated yet.")
        (xpa, xva), (xpb, xvb) = self.x_refs
        if self.x_log:
            la, lb = np.log10(xva), np.log10(xvb)
            x = 10 ** (la + (px - xpa) * (lb - la) / (xpb - xpa))
        else:
            x = xva + (px - xpa) * (xvb - xva) / (xpb - xpa)
        (ypa, yva), (ypb, yvb) = self.y_refs
        if self.y_log:
            la, lb = np.log10(yva), np.log10(yvb)
            y = 10 ** (la + (py - ypa) * (lb - la) / (ypb - ypa))
        else:
            y = yva + (py - ypa) * (yvb - yva) / (ypb - ypa)
        return x, y

    def _data_to_pixel(self, x, y):
        (xpa, xva), (xpb, xvb) = self.x_refs
        if self.x_log:
            la, lb = np.log10(xva), np.log10(xvb)
            px = xpa + (np.log10(x) - la) * (xpb - xpa) / (lb - la)
        else:
            px = xpa + (x - xva) * (xpb - xpa) / (xvb - xva)
        (ypa, yva), (ypb, yvb) = self.y_refs
        if self.y_log:
            la, lb = np.log10(yva), np.log10(yvb)
            py = ypa + (np.log10(y) - la) * (ypb - ypa) / (lb - la)
        else:
            py = ypa + (y - yva) * (ypb - ypa) / (yvb - yva)
        return px, py

    # ---------- step 1: inspect ----------
    def show(self):
        self._open(f"{self.image_path.name}  (x_log={self.x_log}, y_log={self.y_log})")

    # ---------- step 2: calibrate ----------
    def calibrate(self, x_refs, y_refs):
        """Click 4 reference points in order: x[0], x[1], y[0], y[1]."""
        for is_log, refs, name in [(self.x_log, x_refs, "x"), (self.y_log, y_refs, "y")]:
            if is_log and (refs[0] <= 0 or refs[1] <= 0):
                raise ValueError(f"{name}-axis is log: reference values must be > 0.")
        self._mode = "calibrate"
        self._pending = []
        order = [f"x = {x_refs[0]}", f"x = {x_refs[1]}",
                 f"y = {y_refs[0]}", f"y = {y_refs[1]}"]
        fig, ax = self._open(f"Calibrate (1/4): click {order[0]}")

        def on_click(event):
            if event.inaxes != ax or event.xdata is None:
                return
            i = len(self._pending)
            self._pending.append((event.xdata, event.ydata))
            colour = "tab:red" if i < 2 else "tab:blue"
            ax.plot(event.xdata, event.ydata, marker="+", ms=18, mew=2.4, color=colour)
            ax.annotate(order[i], (event.xdata, event.ydata),
                        xytext=(8, -8), textcoords="offset points",
                        fontsize=9, color=colour)
            if i + 1 < 4:
                ax.set_title(f"Calibrate ({i+2}/4): click {order[i+1]}", fontsize=11)
            else:
                xpa, _ = self._pending[0]
                xpb, _ = self._pending[1]
                _, ypa = self._pending[2]
                _, ypb = self._pending[3]
                self.x_refs = [(xpa, x_refs[0]), (xpb, x_refs[1])]
                self.y_refs = [(ypa, y_refs[0]), (ypb, y_refs[1])]
                ax.set_title("Axes calibrated -- ready to extract trajectories.",
                             fontsize=11, color="tab:green")
                fig.canvas.mpl_disconnect(self._cid)
                self._cid = None
                self._mode = None
            fig.canvas.draw_idle()

        self._connect(on_click)

    # ---------- step 3: trajectories ----------
    def start_trajectory(self, name):
        if self.x_refs is None or self.y_refs is None:
            raise RuntimeError("Calibrate the axes first.")
        self._mode = "trajectory"
        self._pending = []
        self._pending_meta = name
        fig, ax = self._open(
            f"Trajectory '{name}': left-click each observed point, right-click to undo. "
            f"Then run finish_trajectory()."
        )
        for tname, pts in self.trajectories.items():
            xs, ys = zip(*[self._data_to_pixel(x, y) for x, y in pts])
            ax.plot(xs, ys, "o", ms=5, mfc="none", mec="gray", alpha=0.4)

        def redraw():
            ax.cla()
            ax.imshow(self.img)
            ax.set_xticks([]); ax.set_yticks([])
            for tname, pts in self.trajectories.items():
                xs, ys = zip(*[self._data_to_pixel(x, y) for x, y in pts])
                ax.plot(xs, ys, "o", ms=5, mfc="none", mec="gray", alpha=0.4)
            if self._pending:
                xs, ys = zip(*self._pending)
                ax.plot(xs, ys, "o", ms=8, mfc="none", mec="red", mew=2)
            ax.set_title(
                f"Trajectory '{name}': {len(self._pending)} points "
                f"(right-click undo, then finish_trajectory())", fontsize=11,
            )
            fig.canvas.draw_idle()

        def on_click(event):
            if event.inaxes != ax or event.xdata is None:
                return
            if event.button == 3:
                if self._pending:
                    self._pending.pop()
                    redraw()
                return
            self._pending.append((event.xdata, event.ydata))
            ax.plot(event.xdata, event.ydata, "o", ms=8, mfc="none", mec="red", mew=2)
            ax.set_title(
                f"Trajectory '{name}': {len(self._pending)} points "
                f"(right-click undo, then finish_trajectory())", fontsize=11,
            )
            fig.canvas.draw_idle()

        self._connect(on_click)

    def finish_trajectory(self):
        if self._mode != "trajectory":
            raise RuntimeError("No trajectory in progress.")
        name = self._pending_meta
        pts = sorted(
            (self.pixel_to_data(px, py) for px, py in self._pending),
            key=lambda p: p[0],
        )
        self.trajectories[name] = pts
        if self._cid is not None:
            self._fig.canvas.mpl_disconnect(self._cid)
            self._cid = None
        self._ax.set_title(
            f"Trajectory '{name}' captured: {len(pts)} points.",
            color="tab:green", fontsize=11,
        )
        self._fig.canvas.draw_idle()
        self._mode = None
        return pts

    # ---------- step 4 / 5: export and verify ----------
    def to_dataframe(self):
        rows = [
            {"trajectory": name, "x": x, "y": y}
            for name, pts in self.trajectories.items()
            for x, y in pts
        ]
        return pd.DataFrame(rows)

    def save_csv(self, path):
        df = self.to_dataframe()
        df.to_csv(path, index=False)
        return df

    def replot(self, ax=None, x_label="x", y_label="y"):
        if ax is None:
            _, ax = plt.subplots(figsize=(8, 6))
        for name, pts in self.trajectories.items():
            xs, ys = zip(*pts)
            ax.plot(xs, ys, "o-", label=name, alpha=0.85)
        if self.x_log:
            ax.set_xscale("log")
        if self.y_log:
            ax.set_yscale("log")
        ax.set_xlabel(x_label)
        ax.set_ylabel(y_label)
        ax.grid(True, which="both", ls=":", alpha=0.5)
        ax.legend(fontsize=8, loc="best")
        ax.set_title("Re-plotted from extracted CSV")
        return ax

## 1. Configure and inspect

Pick the image and declare the axis scales. Each image gets one CSV of the same name in
`extracted_data/`, so figures and their extracted data stay paired.

The two example figures in this repo, one per axis-scale combination:

| Image | x-axis | y-axis | Curves | Extracted to |
|---|---|---|---|---|
| `images/paclitaxel_example.webp` | linear, 0 - 50 hr | **log**, 1 - 10,000 ug/L | one per dose level | `extracted_data/paclitaxel_example.csv` |
| `images/cisplatin_example.png` | **log**, 0.25 - 32 h | linear, 0.0 - 1.0 ug/mL | SC, HITHOC, HIPEC | `extracted_data/cisplatin_example.csv` |

Run the cell and check that the figure renders before going on.

In [ ]:
# --- paclitaxel example (linear x, log y) ---
IMAGE_PATH = "images/paclitaxel_example.webp"
X_LOG = False
Y_LOG = True
X_LABEL = "Time (hr)"
Y_LABEL = "Concentration of paclitaxel (ug/L)"

# --- cisplatin example (log x, linear y) ---
# IMAGE_PATH = "images/cisplatin_example.png"
# X_LOG = True
# Y_LOG = False
# X_LABEL = "Elapsed time (h)"
# Y_LABEL = "Cisplatin concentration (ug/mL)"

# One CSV per image, named after the image.
OUTPUT_CSV = Path("extracted_data") / (Path(IMAGE_PATH).stem + ".csv")
OUTPUT_CSV.parent.mkdir(exist_ok=True)

extractor = PKFigureExtractor(IMAGE_PATH, x_log=X_LOG, y_log=Y_LOG)
extractor.show()

## 2. Calibrate the axes

Pass two known tick locations on each axis, then click the figure four times in this exact
order:

1. the x-axis tick at `x_refs[0]`
2. the x-axis tick at `x_refs[1]`
3. the y-axis tick at `y_refs[0]`
4. the y-axis tick at `y_refs[1]`

Pick references that are far apart and unambiguously labelled (major gridlines are ideal).
On a log axis both values must be > 0. The title tells you which click is expected next, and
turns green when all four are in.

Suggested values:

- `paclitaxel_example`: `x_refs=(0, 50)`, `y_refs=(1, 10_000)`
- `cisplatin_example`: `x_refs=(0.25, 32)`, `y_refs=(0.0, 1.0)`

In [ ]:
extractor.calibrate(x_refs=(0, 50), y_refs=(1, 10_000))

# cisplatin_example: extractor.calibrate(x_refs=(0.25, 32), y_refs=(0.0, 1.0))

## 3. Extract each trajectory

For every curve in the figure:

1. Run `start_trajectory("<name>")`. The figure re-opens; already-captured points show as faint
   grey rings so you can tell which curves are done.
2. Left-click the centre of each observed marker. Right-click to undo the last click.
3. Run `finish_trajectory()` to commit the curve. Points are stored sorted by x.

Name each trajectory exactly as the figure legend does — that name becomes the `trajectory`
column in the CSV. Duplicate the pair of cells below for as many curves as the figure has
(`60 mg/m2`, `180 mg/m2`, `360 mg/m2` for the paclitaxel example; `SC`, `HITHOC`, `HIPEC` for
the cisplatin one).

In [ ]:
extractor.start_trajectory("60 mg/m2")

In [ ]:
extractor.finish_trajectory()

In [ ]:
extractor.start_trajectory("180 mg/m2")

In [ ]:
extractor.finish_trajectory()

In [ ]:
extractor.start_trajectory("360 mg/m2")

In [ ]:
extractor.finish_trajectory()

## 4. Save to CSV

Writes one tidy row per observed point: `trajectory, x, y`.

In [ ]:
df = extractor.save_csv(OUTPUT_CSV)
print(f"Saved {len(df)} points across {df['trajectory'].nunique()} trajectories to {OUTPUT_CSV}")
df.head(20)

## 5. Re-plot from the CSV

Round-trip check: load the CSV back from disk and reconstruct the plot. If the calibration was
right, the reconstruction should sit on top of the original figure. If it does not, the usual
culprit is a mis-clicked calibration point or a wrong `X_LOG` / `Y_LOG` flag — fix it and redo
step 2 onwards.

In [ ]:
loaded = pd.read_csv(OUTPUT_CSV)

fig, ax = plt.subplots(figsize=(8, 6))
for name, group in loaded.groupby("trajectory", sort=False):
    ax.plot(group["x"], group["y"], "o-", label=name, alpha=0.85)
if X_LOG:
    ax.set_xscale("log")
if Y_LOG:
    ax.set_yscale("log")
ax.set_xlabel(X_LABEL)
ax.set_ylabel(Y_LABEL)
ax.grid(True, which="both", ls=":", alpha=0.5)
ax.legend(fontsize=8, loc="best")
ax.set_title(f"Reconstructed from {OUTPUT_CSV.name}")
fig.tight_layout()